# Load Excel data into SQL Server

Loads the 3 sheets of `customer_churn_data_10k.xlsx` (synthetic dataset) into SQL Server as 3 raw tables in `churn_db`:
`db_customer`, `db_subscription`, `db_support`.

Prerequisite: create the database first in SSMS with `CREATE DATABASE churn_db;`

In [1]:
%pip install pandas sqlalchemy pymssql openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sqlalchemy import create_engine

# Config: use the server name shown on the SSMS login screen
SERVER = r"SANDEEP\SQLEXPRESS"
DATABASE = "churn_db"
EXCEL_FILE = "customer_churn_data_10k.xlsx"

In [3]:
# Read all 3 sheets (sheet name -> DataFrame)
sheets = pd.read_excel(EXCEL_FILE, sheet_name=None)

for name, df in sheets.items():
    print(name, df.shape)

db_customer (10040, 8)
db_subscription (10000, 11)
db_support (4005, 6)


In [4]:
# Connect to SQL Server (Windows authentication, no password stored)
engine = create_engine(f"mssql+pymssql://{SERVER}/{DATABASE}")

# Each sheet becomes one raw table; raw tables are never modified later
for name, df in sheets.items():
    df.to_sql(name, engine, if_exists="replace", index=False)
    print(name, len(df), "rows loaded")

db_customer 10040 rows loaded
db_subscription 10000 rows loaded
db_support 4005 rows loaded


In [5]:
# Verify row counts in SQL Server (expected: 10040, 10000, 4005)
for t in ["db_customer", "db_subscription", "db_support"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", engine)["n"][0]
    print(t, n)

db_customer 10040
db_subscription 10000
db_support 4005
